In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from tracking_step_sme_internal import TrackingStepSMEInternal
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_tracking_step_sme(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix, tracking_step: TrackingStepSMEInternal
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema=(
        #        "TrackingStepId string,"
        #        "SourceSystem string,"
        #        "PortfolioKey string,"
        #        "PortfolioName string,"
        #        "IngestionTimestamp timestamp,"
        #        "BatchId string,"
        #        "TrackingStepKey string,"
        #        "StepKey string,"
        #        "StrategyKey int,"
        #        "RealEstateKey string,"
        #        "StartDate date,"
        #        "TargetEndDate date,"
        #        "CreationDate timestamp,"
        #        "IsPrimaryFlag boolean,"
        #        "IsDeletedFlag boolean,"
        #        "RecoveryAmount decimal(18,4),"
        #        "ExpenseAmount decimal(18,4),"
        #        "NumberOfPastAuctions int,"
        #        "LastAuctionDate date,"
        #        "NextAuctionDate date,"
        #        "NextAuctionPrice decimal(18,4),"
        #        "Comments string,"
        #         "SolutionREStatus string,"
        #        "BusinessDate date"
        #        )

        cluster_columns = ["BusinessDate"]
        curated_table_path = f"{curated_zone}.TrackingStep"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "sme",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== sme_trackingstep ==================

        trackingstep_selected_fields = [
            "DebtorStatusReColActionID",
            "Step_id",
            "DebtorStatusReColPropertyKey",
            "DebtorStatusReColActionStrategyID",
            "DebtorStatusReColActionStartDate",
            "DebtorStatusReColActionTargetEndDate",
            "DebtorStatusReColActionCreationDate",
            "DebtorStatusReColActionIsPrimary",
            "DebtorStatusReColActionIsDeleted",
            "DebtorStatusReColActionRecoveryAmount",
            "DebtorStatusReColActionExpenseAmount",
            "DebtorStatusReColActionNumOfPastAuctions",
            "DebtorStatusReColActionLastAuctionDate",
            "DebtorStatusReColActionNextAuctionDate",
            "DebtorStatusReColActionNextAuctionPrice",
            "DebtorStatusReColActionComments",
            "PortfolioID",
            "Step_status",
            "BusinessDate",
        ]

        trackingstep = tracking_step.trackingStep()
        trackingstep = trackingstep.select(trackingstep_selected_fields)

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "portfolio_id_sme",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================ Joins ================

        trackingstep_df = trackingstep.join(
            portfolio,
            portfolio["portfolio_id_sme"] == trackingstep["PortfolioID"],
            "left",
        )

        TrackingStepId_primary_key_fields = [
            trackingstep_df.DebtorStatusReColActionID,
            lit("sme"),
        ]

        trackingstep_df = trackingstep_df.withColumn(
            "TrackingStepId",
            sha2(
                concat_ws(";", *TrackingStepId_primary_key_fields),
                256,
            ),
        ).withColumn("TrackingStepKey", trackingstep.DebtorStatusReColActionID)

        # ================== Selection of Final Columns ==================

        trackingstep_sme = trackingstep_df.select(
            trackingstep_df.TrackingStepId.cast("String").alias("TrackingStepId"),
            lit("sme").alias("SourceSystem"),
            trackingstep.PortfolioID.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("String").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias("IngestionTimestamp"),
            lit(None).alias("BatchId"),
            trackingstep_df.TrackingStepKey.cast("String").alias("TrackingStepKey"),
            trackingstep.Step_id.cast("String").alias("StepKey"),
            trackingstep_df.DebtorStatusReColActionStrategyID.cast("Int").alias("StrategyKey"),
            trackingstep_df.DebtorStatusReColPropertyKey.cast("String").alias("RealEstateKey"),
            trackingstep_df.DebtorStatusReColActionStartDate.cast("Date").alias("StartDate"),
            trackingstep_df.DebtorStatusReColActionTargetEndDate.cast("Date").alias("TargetEndDate"),
            trackingstep_df.DebtorStatusReColActionCreationDate.cast("Timestamp").alias("CreationDate"),
            trackingstep_df.DebtorStatusReColActionIsPrimary.cast("Boolean").alias("IsPrimaryFlag"),
            trackingstep_df.DebtorStatusReColActionIsDeleted.cast("Boolean").alias("IsDeletedFlag"),
            trackingstep_df.DebtorStatusReColActionRecoveryAmount.cast("Decimal(18,4)").alias("RecoveryAmount"),
            trackingstep_df.DebtorStatusReColActionExpenseAmount.cast("Decimal(18,4)").alias("ExpenseAmount"),
            trackingstep_df.DebtorStatusReColActionNumOfPastAuctions.cast("Int").alias("NumberOfPastAuctions"),
            trackingstep_df.DebtorStatusReColActionLastAuctionDate.cast("Date").alias("LastAuctionDate"),
            trackingstep_df.DebtorStatusReColActionNextAuctionDate.cast("Date").alias("NextAuctionDate"),
            trackingstep_df.DebtorStatusReColActionNextAuctionPrice.cast("Decimal(18,4)").alias("NextAuctionPrice"),
            trackingstep_df.DebtorStatusReColActionComments.cast("String").alias("Comments"),
            trackingstep_df.Step_status.cast("String").alias("SolutionREStatus"),
            trackingstep_df.BusinessDate.cast("Date").alias("BusinessDate"),
        )

        # ========================== Append ==========================
        trackingstep_sme.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")
    logging.info("We did it!")


In [0]:
tracking_step = TrackingStepSMEInternal(spark, catalog_landed, run_date)

In [0]:
load_curated_tracking_step_sme(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix, tracking_step)